# Stage 1.4.2.5.3 — Docling Layout & Reading Order

## Objective
We already learned:
Stage 1.4.2.5.1 — Basic OCR

      Image
      ↓
      OCR
      ↓
      Text

Then:
Stage 1.4.2.5.2 — Docling for Scanned PDFs

      Scanned PDF
         ↓
      Docling
         ↓
      DoclingDocument
         ↓
      Markdown / structured representation

Now we're asking a deeper question:

Can Docling understand the layout of a document and determine the correct logical reading order?

## Step 1 — Understand the Problem
Consider a document like this:

      ┌──────────────────────┬──────────────────────┐
      │ Heading A            │ Heading B            │
      │                      │                      │
      │ Paragraph A1         │ Paragraph B1         │
      │ Paragraph A2         │ Paragraph B2         │
      │                      │                      │
      └──────────────────────┴──────────────────────┘

A naive text extractor might produce:

      Heading A
      Heading B
      Paragraph A1
      Paragraph B1
      Paragraph A2
      Paragraph B2

But the logical reading order might be:

      Heading A
      Paragraph A1
      Paragraph A2

      Heading B
      Paragraph B1
      Paragraph B2

That's the problem layout and reading-order analysis tries to solve.

## Step 2 — Understand Why This Matters to RAG
This is extremely important.
Suppose a PDF contains:

      Column 1                  Column 2

      Azure Event Hubs          Azure Data Explorer

      Event ingestion           Analytics

      High-volume events        Query processing

If the extraction order is wrong, we might create a chunk such as:

      Azure Event Hubs
      Azure Data Explorer
      Event ingestion
      Analytics
      High-volume events
      Query processing

The text exists, but its relationships have been damaged.
That can negatively affect:

      Chunking
         ↓
      Embedding
         ↓
      Retrieval
         ↓
      LLM context

Therefore:

Good RAG begins with good document understanding.

## Step 3 — Locate the Sample PDF
Download the sample PDF above and place it somewhere accessible from your notebook.
For example:

      rag-learning/
      │
      ├── notebooks/
      │
      ├── data/
      │   └── docling_layout_reading_order_sample.pdf
      │
      └── ...

Then:

In [5]:
from pathlib import Path

pdf_path = Path("D:/AI Learning/rag-learning/data/raw/pdf/docling_layout_reading_order_sample.pdf"
)

print(pdf_path.exists())
print(pdf_path)

True
D:\AI Learning\rag-learning\data\raw\pdf\docling_layout_reading_order_sample.pdf


Adjust the path according to where you saved it.
We want:

True

## Step 4 — Check Your Docling Version
We're using your installed:

Docling 2.120.3

Let's confirm from the notebook:

In [6]:
import importlib.metadata
print(importlib.metadata.version("docling"))

2.120.3


Expected:

2.120.3

## Step 5 — Create the Document Converter
Use the same approach from Stage 1.4.2.5.2:

In [7]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

d:\AI Learning\rag-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 6 — Convert the Sample PDF
Run:

In [8]:
result = converter.convert(pdf_path)

print(result.status)

[INFO] 2026-08-19 23:38:16,369 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 23:38:16,390 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 23:38:16,460 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 23:38:16,462 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 23:38:16,853 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 23:38:16,855 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 23:38:16,860 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-19 23:38:16,861 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobil

ConversionStatus.SUCCESS


We want a successful conversion.

## Step 7 — Get the DoclingDocument

In [9]:
doc = result.document

print(type(doc))

<class 'docling_core.types.doc.document.DoclingDocument'>


You should get a DoclingDocument.

## Step 8 — Export to Markdown
This is our first way to observe the result:

In [10]:
markdown_text = doc.export_to_markdown()

print(markdown_text)

## Azure Event Processing Architecture

This sample document is designed for Stage 1.4.2.5.3 - Docling Layout &amp; Reading Order. It contains headings, paragraphs, a two-column layout, a table, a caption, and content whose logical reading order differs from simple top-to-bottom visual scanning.

## 1. Overview

An enterprise application publishes events to Azure Event Hubs. A downstream processing component consumes those events and sends selected records to Azure Data Explorer for analytics. The architecture contains independent ingestion and analytics stages.

## Ingestion Layer

The producer sends events to Event Hubs. Event Hubs provides scalable event ingestion and partitions the event stream for parallel consumption.

Key responsibility: reliably accept high-volume event data.

Figure 1 - Logical processing flow

## 2. Processing Stages

|   Stage | Component           | Purpose                              |
|---------|---------------------|-------------------------------------

What are we looking for?
Look carefully at:

        Heading order
        Paragraph order
        Two-column content
        Table position
        Caption position
        Page transitions

Don't just ask:

        "Did Docling extract the words?"

Ask:

        "Did Docling preserve the logical structure?"

That's the purpose of this stage.

## Step 9 — Inspect the Page Structure
Now we want to go deeper than Markdown.
The DoclingDocument contains structured information about document elements.
Let's first inspect what is available:

In [11]:
print(type(doc))

<class 'docling_core.types.doc.document.DoclingDocument'>


In [12]:
[m for m in dir(doc) if not m.startswith("_")]

['add_code',
 'add_comment',
 'add_document',
 'add_field_heading',
 'add_field_hint',
 'add_field_item',
 'add_field_key',
 'add_field_region',
 'add_field_value',
 'add_form',
 'add_formula',
 'add_group',
 'add_heading',
 'add_inline_group',
 'add_key_values',
 'add_list_group',
 'add_list_item',
 'add_marker',
 'add_node_items',
 'add_ordered_list',
 'add_page',
 'add_picture',
 'add_table',
 'add_table_cell',
 'add_text',
 'add_title',
 'add_unordered_list',
 'append_child_item',
 'body',
 'check_version_is_compatible',
 'concatenate',
 'construct',
 'copy',
 'delete_items',
 'delete_items_range',
 'dict',
 'export_to_dict',
 'export_to_doclang',
 'export_to_doctags',
 'export_to_document_tokens',
 'export_to_element_tree',
 'export_to_html',
 'export_to_markdown',
 'export_to_text',
 'export_to_vtt',
 'extract_items_range',
 'field_items',
 'field_regions',
 'filter',
 'form_items',
 'from_orm',
 'furniture',
 'get_visualization',
 'groups',
 'insert_code',
 'insert_document',
 '

This is useful because we're working specifically with Docling 2.120.3, rather than blindly copying APIs from another version.

## Step 10 — Inspect Document Items
For this stage, one particularly useful concept is the document's items.
Try:

In [13]:
print(doc.body)

self_ref='#/body' parent=None children=[RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2'), RefItem(cref='#/texts/3'), RefItem(cref='#/texts/4'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/tables/0'), RefItem(cref='#/texts/9'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/texts/12'), RefItem(cref='#/texts/13'), RefItem(cref='#/texts/14'), RefItem(cref='#/texts/15'), RefItem(cref='#/texts/16'), RefItem(cref='#/tables/1'), RefItem(cref='#/texts/17')] content_layer=<ContentLayer.BODY: 'body'> meta=None name='_root_' label=<GroupLabel.UNSPECIFIED: 'unspecified'>


In [14]:
print(doc.__dict__.keys())

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'field_regions', 'field_items', 'pages'])


We're looking for how Docling 2.120.3 represents the document internally.
Depending on the exact object structure exposed by your installed version, we'll inspect the relevant collections rather than assuming a particular API.

## Step 11 — Why We Are Inspecting the Structure
Imagine Docling internally recognizes something like:

        Page 1
        │
        ├── Heading
        ├── Paragraph
        ├── Paragraph
        ├── Heading
        ├── Paragraph
        ├── Table
        └── Caption

That's much more useful than:

    one giant string

because later we can make intelligent decisions about chunking.
For example:

        Heading
            +
        Paragraphs
            ↓
        one semantic chunk

rather than blindly doing:

    every 500 characters

## Step 12 — Inspect the Markdown More Carefully
Let's print the first page's extracted Markdown separately if possible, but first simply inspect:

In [15]:
print(markdown_text[:5000])

## Azure Event Processing Architecture

This sample document is designed for Stage 1.4.2.5.3 - Docling Layout &amp; Reading Order. It contains headings, paragraphs, a two-column layout, a table, a caption, and content whose logical reading order differs from simple top-to-bottom visual scanning.

## 1. Overview

An enterprise application publishes events to Azure Event Hubs. A downstream processing component consumes those events and sends selected records to Azure Data Explorer for analytics. The architecture contains independent ingestion and analytics stages.

## Ingestion Layer

The producer sends events to Event Hubs. Event Hubs provides scalable event ingestion and partitions the event stream for parallel consumption.

Key responsibility: reliably accept high-volume event data.

Figure 1 - Logical processing flow

## 2. Processing Stages

|   Stage | Component           | Purpose                              |
|---------|---------------------|-------------------------------------

Look for something like:

# Azure Event Processing Architecture

## 1. Overview

...

## 2. Processing Stages

        ...

        | Stage | Component | Purpose |
        |---|---|---|
        ...

The exact result will depend on how Docling interprets the generated PDF.

## Step 13 — Focus on Reading Order
Our sample contains this logical sequence:

        1. Overview

        2. Ingestion Layer

        3. Analytics Layer

        4. Processing Stages

        5. Important Design Considerations

We intentionally designed the document so that the visual layout isn't simply a single linear stream.
The question we're testing is:

        Visual position
            ≠
        Logical reading order

A document-understanding system needs to infer the latter.

## Step 14 — Compare with a Naive Text Extraction Approach
This comparison is useful.

        Naive PDF extraction
        PDF
        ↓
        Text extraction
        ↓
        String

Potential problem:

        Wrong ordering
        Lost relationships
        Lost layout
        Lost table structure

Docling

        PDF
        ↓
        Document understanding
        ↓
        Layout
        ↓
        Reading order
        ↓
        Structured document

Potentially:

        Heading
        ↓
        Paragraph
        ↓
        Paragraph
        ↓
        Table
        ↓
        Caption

That's why we're learning Docling.

## Step 15 — Understand Reading Order in RAG
Suppose our document says:

    Azure Event Hubs
and underneath it:

    Event ingestion platform

Then in another column:

    Azure Data Explorer
with:

Analytics platform

If our parser mixes them up, the resulting embedding might represent a distorted relationship.
Instead, we want:

        Azure Event Hubs
            ↓
        Event ingestion platform

and:

        Azure Data Explorer
            ↓
        Analytics platform

This produces much better semantic units for subsequent chunking.

## Step 16 — Understand the Relationship with Chunking
This is one of the most important lessons from this stage.
We previously learned that chunking strategy matters.
But now notice:

        Document Understanding
                ↓
        Layout
                ↓
        Reading Order
                ↓
        Semantic Structure
                ↓
        Chunking

Therefore:

    You shouldn't think of chunking as an isolated operation.
    
The quality of your chunks depends partly on how well you understand the source document.
This is why modern RAG ingestion pipelines increasingly use document-understanding frameworks.

## Step 17 — Inspect the Table
Our sample PDF contains a table:

Stage | Component | Purpose

After conversion:

In [16]:

markdown_text = doc.export_to_markdown()

print(markdown_text)

markdown_path = Path("output/stage1.4.2.5.3_scanned_document_docling.md")
# Create the parent directory (and any missing intermediate folders) if it doesn't exist
markdown_path.parent.mkdir(parents=True, exist_ok=True)
markdown_path.write_text(markdown_text,encoding="utf-8")

print(f"Saved: {markdown_path}")

## Azure Event Processing Architecture

This sample document is designed for Stage 1.4.2.5.3 - Docling Layout &amp; Reading Order. It contains headings, paragraphs, a two-column layout, a table, a caption, and content whose logical reading order differs from simple top-to-bottom visual scanning.

## 1. Overview

An enterprise application publishes events to Azure Event Hubs. A downstream processing component consumes those events and sends selected records to Azure Data Explorer for analytics. The architecture contains independent ingestion and analytics stages.

## Ingestion Layer

The producer sends events to Event Hubs. Event Hubs provides scalable event ingestion and partitions the event stream for parallel consumption.

Key responsibility: reliably accept high-volume event data.

Figure 1 - Logical processing flow

## 2. Processing Stages

|   Stage | Component           | Purpose                              |
|---------|---------------------|-------------------------------------

In [17]:
import json
from pathlib import Path

# convert the document to a json file
json_path = Path("output/stage1.4.2.5.3 docling_layout_reading_order_sample.json")
# Create the parent directory (and any missing intermediate folders) if it doesn't exist
json_path.parent.mkdir(parents=True, exist_ok=True)
doc_dict = doc.export_to_dict()
with open(json_path, "w") as f:
    json.dump(doc_dict, f,indent=2, ensure_ascii=False)

Look for a Markdown table.
If Docling preserves it as:

| Stage | Component | Purpose |
|---|---|---|
| 1 | Event Producer | ... |
| 2 | Azure Event Hubs | ... |

that's an important observation.
Compare that with basic OCR, which could produce:

Stage Component Purpose
1 Event Producer Publishes...
2 Azure Event Hubs Ingests...

The second output contains the words but may have lost the explicit table relationships.

## Step 18 — The Architecture We Are Building Toward
Our ingestion architecture is gradually becoming:

                   Enterprise PDF
                         │
                         ▼
                      Docling
                         │
          ┌──────────────┼──────────────┐
          │              │              │
          ▼              ▼              ▼
         OCR           Layout         Tables
          │              │              │
          └──────────────┼──────────────┘
                         ▼
                 DoclingDocument
                         │
                         ▼
                Reading Order
                         │
                         ▼
                Structured Content
                         │
                         ▼
                    Chunking
                         │
                         ▼
                    Embedding
                         │
                         ▼
                  Vector Store

This is the conceptual reason we're doing this stage.

## Step 19 — What We Are NOT Doing Yet
Don't worry about:

OCR engine tuning
Tesseract configuration
image preprocessing
formula enrichment
picture description
multimodal embeddings
table-specific chunking
LangChain integration

Those are separate concerns.
Our question right now is simply:

Can Docling correctly understand the spatial organization and logical reading order of our document?

## Step 20 — Your Hands-On Experiment
Run these cells in order.

In [18]:
from pathlib import Path
from docling.document_converter import DocumentConverter

In [19]:
pdf_path = Path("D:/AI Learning/rag-learning/data/raw/pdf/docling_layout_reading_order_sample.pdf")
print("Exists:", pdf_path.exists())

Exists: True


In [20]:
converter = DocumentConverter()

In [21]:
result = converter.convert(pdf_path)

print("Status:", result.status)

[INFO] 2026-08-19 23:38:36,358 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 23:38:36,360 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 23:38:36,416 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 23:38:36,421 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 23:38:36,834 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 23:38:36,837 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 23:38:36,841 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-19 23:38:36,842 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobil

Status: ConversionStatus.SUCCESS


In [22]:
doc = result.document

print(type(doc))

<class 'docling_core.types.doc.document.DoclingDocument'>


In [23]:
markdown_text = doc.export_to_markdown()

print(markdown_text)

## Azure Event Processing Architecture

This sample document is designed for Stage 1.4.2.5.3 - Docling Layout &amp; Reading Order. It contains headings, paragraphs, a two-column layout, a table, a caption, and content whose logical reading order differs from simple top-to-bottom visual scanning.

## 1. Overview

An enterprise application publishes events to Azure Event Hubs. A downstream processing component consumes those events and sends selected records to Azure Data Explorer for analytics. The architecture contains independent ingestion and analytics stages.

## Ingestion Layer

The producer sends events to Event Hubs. Event Hubs provides scalable event ingestion and partitions the event stream for parallel consumption.

Key responsibility: reliably accept high-volume event data.

Figure 1 - Logical processing flow

## 2. Processing Stages

|   Stage | Component           | Purpose                              |
|---------|---------------------|-------------------------------------

In [24]:
print(doc.__dict__.keys())

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'field_regions', 'field_items', 'pages'])


In [25]:
print([m for m in dir(doc) if not m.startswith("_")])

['add_code', 'add_comment', 'add_document', 'add_field_heading', 'add_field_hint', 'add_field_item', 'add_field_key', 'add_field_region', 'add_field_value', 'add_form', 'add_formula', 'add_group', 'add_heading', 'add_inline_group', 'add_key_values', 'add_list_group', 'add_list_item', 'add_marker', 'add_node_items', 'add_ordered_list', 'add_page', 'add_picture', 'add_table', 'add_table_cell', 'add_text', 'add_title', 'add_unordered_list', 'append_child_item', 'body', 'check_version_is_compatible', 'concatenate', 'construct', 'copy', 'delete_items', 'delete_items_range', 'dict', 'export_to_dict', 'export_to_doclang', 'export_to_doctags', 'export_to_document_tokens', 'export_to_element_tree', 'export_to_html', 'export_to_markdown', 'export_to_text', 'export_to_vtt', 'extract_items_range', 'field_items', 'field_regions', 'filter', 'form_items', 'from_orm', 'furniture', 'get_visualization', 'groups', 'insert_code', 'insert_document', 'insert_form', 'insert_formula', 'insert_group', 'insert_

## The key thing I want you to observe
Don't worry yet about writing a lot of code.
After running the conversion, look at the Markdown output and compare it against the visual PDF.
We're testing three things:

        1. Did Docling recognize the headings?
                ↓
        2. Did it preserve the logical reading order?
                ↓
        3. Did it preserve the table structure?

Once we see your actual output from Docling 2.120.3, we'll inspect the DoclingDocument structure in the next step and learn how Docling represents layout and reading order internally. That is much more valuable than simply calling an export method and moving on.